# In this script we will explore dlis file

In [5]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import logging 
import dlisio 
from dlisio import dlis


In [6]:
filename="/home/ashraf/Downloads/University_of_Utah_MU-ESW1_FMI-HD_7390-7527ft_Run3.dlis"

### Logical files

In [11]:
# With the first logical file stored in f
# All other logical files stored in f_tail

f,*f_tail=dlis.load(filename)

# Check if there is one or more logical files in f_tail
if len(f_tail):logging.warning("There are more logical files in tail")

## Origin

In [13]:
origin, *origin_tail=f.origins

if len(origin_tail): logging.warning("f contains multiple origins")

In [15]:
# Print the description of all origin
for origin in f.origins:
    print(origin.describe())

------
Origin
------
name   : WELL-MU-ESW1
origin : 122
copy   : 0

Logical file ID          : MU-ESW1
File set name and number : MU-ESW1 / 1
File number and type     : 0 / CUSTOMER

Field                   : None
Well (id/name)          :  / MU-ESW1
Produced by (code/name) : 440 / Schlumberger
Produced for            : University of Utah
Created                 : 2019-05-08 11:01:46

Created by              : Techlog, (version: 2013.4.0 (rev: 128578))
Other programs/services : WELL-MU-ESW1


------
Origin
------
name   : DATASET-FMI_HD_7390-7527ft_shift
origin : 123
copy   : 0

Logical file ID          : MU-ESW1
File set name and number : MU-ESW1 / 1
File number and type     : 0 / CUSTOMER

Field                   : None
Well (id/name)          :  / MU-ESW1
Produced by (code/name) : 440 / Schlumberger
Produced for            : University of Utah
Created                 : 2019-05-08 11:01:46

Created by              : Techlog, (version: 2013.4.0 (rev: 128578))
Other programs/services :

# Parameters

In [22]:
# A helper function that extracts specified information from the logical file into a pandas dataframe

def summarize(objs, **kwargs):
    """Create a pd.DataFrame that summarize the content of 'objs', One 
    object pr. row
    
    Parameters
    ----------
    
    objs : list()
        list of metadata objects
        
    **kwargs
        Keyword arguments 
        Use kwargs to tell summarize() which fields (attributes) of the 
        objects you want to include in the DataFrame. The parameter name 
        must match an attribute on the object in 'objs', while the value 
        of the parameters is used as a column name. Any kwargs are excepted, 
        but if the object does not have the requested attribute, 'KeyError' 
        is used as the value.
        
    Returns
    -------
    
    summary : pd.DataFrame
    """

    summary = []
    for attr, label in kwargs.items():
        column=[]
        for obj in objs:
            try:
                value=getattr(obj,attr)
            except AttributeError:
                value ="keyerror"
            
            column.append(value)
        summary.append(column)

    summary=pd.DataFrame(summary).T
    summary.columns=kwargs.values()
    return summary

In [ ]:
# Make a pandas dataframe that contains parameter (constant) details
parameter_table = summarize(f.parameters, name='Name', long_name='Long name', values='Value(s)')

# Add units through the temporary units interface
units_column = []
for i, par in enumerate(f.parameters):
    if i not in parameter_table.index: continue
    try:
        units_column.append(par.attic['VALUES'].units)
    except KeyError:
        units_column.append(None)
parameter_table['Units'] = units_column

# Sort the table alphabetically on parameter name
parameter_table.sort_values('Name')

# Display the first 20 rows of the parameter table
display(parameter_table.head(20))

# Export parameter table to CSV for ease of review
#parameter_table.to_csv('parameter_table.csv', index=False)

,Name,Long name,Value(s),Units
0,TDL,Logger Total Depth,[7546.0],ft
1,PRODUCER-NAME,,[Schlumberger],
2,DFT,Drilling Fluid Type,[WBM],
3,EGL,Elevation of Ground Level,[5536.0],ft
4,DLAB,Date Logger At Bottom,[15-Sep-2017],
5,EPD,Elevation of Permanent Datum (PDAT) above Mean...,[5536.0],ft
6,MRT,Maximum Recorded Temperature,[289.4599914550781],degF
7,MFST,Mud Filtrate Sample Temperature,[74.5],degF
8,RMB,Resistivity of Mud at Bottom Hole Temperature,[0.4027419984340668],ohm.m
9,PRODUCER-CODE,,[440],
